# 02 Correction Validation

This notebook validates the RPF correction models used in the article. It compares the trainable `m8_xgb` model with the deterministic `m7_dtr` benchmark on two tasks: Alpha leave-one-station-out validation and Alpha-to-Beta transfer validation.

**Inputs.** The notebook reads `dataset/final/dataset_alpha.parquet` and `dataset/final/dataset_beta.parquet` through the shared `journal_v2` config.

**Outputs.** It writes correction prediction audit CSVs, primary metric CSVs, a detailed correction summary table, a compact Beta transfer table, confusion-matrix figures, score figures, and a Beta site-level score distribution figure under `outputs/*/02_correction_validation/`.

**Run modes.** By default `execution.run_full_correction_validation` is `false`, so the notebook produces placeholder smoke outputs only. Flip that config value to `true` only when you are ready for the real XGBoost run.


## 1. Imports And Paths

This section resolves the article root, imports the shared helper module, loads the config, and prints the run mode. The helper module contains the heavy experiment logic so the notebook remains readable, but every major helper call below explains what it does internally.


In [ ]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

print(f"Full correction validation: {cfg['execution']['run_full_correction_validation']}")
print(f"Reuse correction prediction files: {cfg['execution'].get('reuse_correction_prediction_files', False)}")


## 2. Preflight Readiness Check

`h.correction_validation_preflight()` performs a no-training readiness check. It loads the final datasets, confirms dependency availability, recomputes the Alpha LOSO sites, ranks all Beta sites by current RPF labels, and reports the expected full-run workload. This is the safest cell to run before switching the config into full mode.


In [ ]:
# This helper does not train models or write experiment outputs.
preflight = h.correction_validation_preflight(article_root)
display(preflight["readiness"])
display(preflight["dependencies"])
display(preflight["workload"])
display(preflight["expected_outputs"])


## 3. Confirm Inputs, Folds, And Beta Site Order

The fold sites are recomputed from Alpha labels, and the Beta site order is recomputed from the current Beta labels. In the full run, Beta predictions are made once per method for the whole Beta dataset; overall and per-site metrics are then calculated downstream from those same prediction frames.


In [ ]:
# Load final datasets once here for visible sanity checks before the workflow writes outputs.
alpha = h.load_dataset(article_root, cfg, "alpha")
beta = h.load_dataset(article_root, cfg, "beta")
alpha_sites = h.alpha_loso_sites(alpha, cfg)
print(f"Alpha LOSO sites: {alpha_sites}")
print(f"Beta rows: {len(beta):,}")
display(preflight["beta_rankings"][["substation_id", "rpf_days", "rpf_intervals", "rpf_day_pct"]])


## 4. Run Correction Validation

`h.run_correction_validation()` is the main workflow. In smoke mode it writes deterministic placeholder metrics and journal-style figures for layout checking. In full mode it performs exactly four `m8_xgb` trainings: three Alpha LOSO folds and one Alpha-to-Beta transfer model. Beta `m8_xgb` predictions are generated once for all sites, and Beta overall plus all-site metrics are calculated from that one prediction frame. Alpha `m7_dtr` inference is batched over the three held-out test folds to avoid repeated deterministic scans.

Public outputs use `day` and `interval` levels. The `interval` level is scoped to the configured daytime interval window, currently 06:00-18:00.


In [ ]:
# Smoke mode writes placeholder outputs; full mode trains/evaluates the configured correction models.
result = h.run_correction_validation(article_root)
print(result["status"])
result["metrics"].head(20)


## 5. Inspect Beta Transfer And All-Site Rows

The Beta transfer rows summarise the full Beta dataset. The Beta site rows break the same prediction results into one row per site, method, and level, which supports the site-level box-and-whisker figure without rerunning either model.


In [ ]:
metrics = result["metrics"]
beta_transfer = metrics.loc[metrics["dataset"] == "Beta"].copy()
beta_site_metrics = result["beta_site_metrics"]

display(beta_transfer)
display(beta_site_metrics)
